# 01E - Two-Minute Rotating YOLO Training

This is the accuracy-focused successor to the preserved executed `01D` notebook. It
continues from the strongest `01C` checkpoint, trains at 576 pixels, and uses a fresh
class-balanced sample from a broad scene pool every epoch. The default RTX 3050
profile was measured to target approximately two minutes per epoch.

The notebook reports object-detection metrics correctly: precision, recall, F1,
mAP50, and mAP50:95. A high displayed confidence is an inference threshold, not
classification accuracy.


## What the recorded runs actually achieved

| Run | Validation size | Precision | Recall | mAP50 | mAP50:95 |
|---|---:|---:|---:|---:|---:|
| `01` frozen baseline | 8,000 | 0.464 | 0.333 | 0.329 | 0.176 |
| `01` fine-tuned baseline | 8,000 | 0.444 | 0.347 | 0.329 | 0.173 |
| `01` held-out test | 2,000 | 0.466 | 0.349 | 0.338 | 0.179 |
| `01C` strongest checkpoint, common 1,200-image validation | 1,200 | 0.521 | 0.368 | **0.367** | **0.190** |
| executed `01D` micro-run | 100 | 0.387 | 0.261 | 0.222 | 0.123 |

`01D` was not just "too fast": it repeatedly trained on only 300 images at 320
pixels. It discarded small-object detail and did not expose the optimizer to enough
new scenes. Its tiny 100-image validation set also made model selection noisy.

The goal here is measurable improvement, not a promised number. Reaching 0.80 mAP50
for six separately scored BDD100K classes on this laptop is an ambitious research
target and cannot be guaranteed by a notebook setting.


## Why this design

- [Ultralytics training documentation](https://docs.ultralytics.com/modes/train/)
  documents AMP, layer freezing, class-frequency weighting, resolution, and
  augmentation controls used here.
- The [Ultralytics fine-tuning guide](https://docs.ultralytics.com/guides/finetuning-guide/)
  recommends AdamW for fewer than 10,000 iterations, `freeze=10` for a small
  COCO-like dataset, lower learning rates, and more resolution for small objects.
- [BDD100K](https://arxiv.org/abs/1805.04687) is deliberately diverse across scene,
  weather, and time conditions, so rotating through distinct images is important.
- A published six-class road detector used 10,000 BDD100K images for
  [100 epochs](https://www.mdpi.com/2227-7390/12/9/1331), while the open-source
  [YOLOPX](https://github.com/jiaoZ7688/YOLOPX) result used the full training set,
  a 32.9M-parameter model, a Tesla V100, and an epoch-195 checkpoint. Those results
  are not comparable to a 300-image micro-run.
- Knowledge distillation is not enabled by default. Ultralytics recommends a
  high-accuracy teacher trained on the same data, and no such six-class teacher
  exists locally. A COCO-only teacher would add training cost without reliable
  BDD100K supervision.

The custom sampler keeps the optimizer and cosine schedule continuous while limiting
each epoch to a fixed image budget. It samples without replacement and gives scenes
containing rare bus and truck labels a higher chance of appearing.


In [ ]:
from pathlib import Path
import json
import random
import shutil
import sys
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import ultralytics
import yaml
from IPython.display import display
from ultralytics import YOLO

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "road_detection").exists():
    raise FileNotFoundError("Start JupyterLab from the Final_Project directory.")
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from road_detection.budgeted_yolo_trainer import make_budgeted_detection_trainer
from road_detection.yolo_training_utils import prepare_fast_data_files

print(f"Project: {PROJECT_ROOT}")
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"Ultralytics: {ultralytics.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

required_ultralytics = (8, 4, 112)
installed_ultralytics = tuple(
    int(part) for part in ultralytics.__version__.split(".")[:3]
)
if installed_ultralytics < required_ultralytics:
    raise RuntimeError("Run: python -m pip install -e .")


## Configuration

The default is the measured RTX 3050 profile. A full two-epoch benchmark with 2,200
training images at 576 pixels measured 139.0 seconds for the warm-up epoch and 104.7
seconds for the steady epoch with 200 validation images. The notebook uses 400
validation images, which adds about four seconds, so steady epochs should remain close
to two minutes and the first epoch should remain below 150 seconds.

Set `RESUME_TRAINING = True` only after an interrupted run with the same `RUN_TAG`.


In [ ]:
SEED = 57
RUN_MODE = "two_minute_rtx3050"
RUN_TAG = "v5_rotating_2min"
RUN_TRAINING = True
RESUME_TRAINING = False

DATA_YAML = PROJECT_ROOT / "data" / "bdd100k_yolo" / "data.yaml"
RUN_ROOT = PROJECT_ROOT / "runs" / "notebooks" / "yolo_accuracy"
MANIFEST_DIR = RUN_ROOT / "manifests" / RUN_TAG
TRAIN_RUN_NAME = f"bdd100k_two_minute_{RUN_TAG}"
TRAIN_RUN_DIR = RUN_ROOT / TRAIN_RUN_NAME

TARGET_SECONDS_PER_EPOCH = 120.0
MAX_ACCEPTABLE_SECONDS_PER_EPOCH = 150.0
TARGET_MAP50 = 0.80
TARGET_F1 = 0.80
TARGET_STRICT_PRECISION = 0.80
STRICT_CONFIDENCE_FLOOR = 0.80
MIN_STRICT_RECALL = 0.01
BALANCED_CONFIDENCE_FLOOR = 0.15

if torch.cuda.is_available():
    DEVICE = "0"
    POOL_SIZE = 12_000
    SAMPLES_PER_EPOCH = 2_200
    EPOCH_VALIDATION_COUNT = 400
    TRAIN_IMGSZ = 576
    EVAL_IMGSZ = 640
    BATCH = 16
    EVAL_BATCH = 16
    FREEZE = 10
    EPOCHS = 240
    WORKERS = 0
else:
    DEVICE = "cpu"
    POOL_SIZE = 2_000
    SAMPLES_PER_EPOCH = 200
    EPOCH_VALIDATION_COUNT = 100
    TRAIN_IMGSZ = 416
    EVAL_IMGSZ = 512
    BATCH = 4
    EVAL_BATCH = 4
    FREEZE = 16
    EPOCHS = 20
    WORKERS = 0

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print({
    "mode": RUN_MODE,
    "pool_size": POOL_SIZE,
    "samples_per_epoch": SAMPLES_PER_EPOCH,
    "train_imgsz": TRAIN_IMGSZ,
    "batch": BATCH,
    "epochs": EPOCHS,
    "device": DEVICE,
})


## Build a broad rotating pool

On this machine the prior `01C` manifest supplies 8,000 class-aware scenes, and this
cell adds 4,000 deterministic fresh scenes. On a new clone it uses the conversion-time
class index when available. Only `SAMPLES_PER_EPOCH` images are drawn from this pool
per epoch, with a new rare-class-aware draw every time.


In [ ]:
if not DATA_YAML.exists():
    raise FileNotFoundError(
        f"Missing {DATA_YAML}. Convert BDD100K with scripts/prepare_bdd100k.ps1 first."
    )

data_files = prepare_fast_data_files(
    data_yaml=DATA_YAML,
    output_dir=MANIFEST_DIR,
    run_root=RUN_ROOT,
    preferred_manifest_tag="v3_fast",
    main_count=POOL_SIZE,
    refine_count=min(2_500, POOL_SIZE),
    validation_count=EPOCH_VALIDATION_COUNT,
    seed=SEED,
)
pool_images = [
    line for line in data_files.main_manifest.read_text(encoding="utf-8").splitlines()
    if line
]
epoch_val_images = [
    line for line in data_files.validation_manifest.read_text(encoding="utf-8").splitlines()
    if line
]
print(f"Data source: {data_files.source}")
print(f"Rotating pool: {len(pool_images):,} images")
print(f"Per-epoch validation: {len(epoch_val_images):,} images")
print(f"Training YAML: {data_files.main_yaml}")


## Select the strongest starting checkpoint

The one-minute `01D` checkpoint is deliberately excluded because its common-holdout
quality is lower. The preferred start is the executed `01C` checkpoint, which reached
0.367 mAP50 on the shared 1,200-image validation sample.


In [ ]:
checkpoint_candidates = [
    RUN_ROOT / "bdd100k_fast_overnight_v3_fast_main" / "weights" / "best.pt",
    RUN_ROOT / "bdd100k_overnight_gpu_v2_main" / "weights" / "best.pt",
    PROJECT_ROOT / "runs" / "notebooks" / "yolo" / "bdd100k_cpu_quick_finetuned" / "weights" / "best.pt",
]
LAST_WEIGHTS = TRAIN_RUN_DIR / "weights" / "last.pt"

if RESUME_TRAINING:
    if not LAST_WEIGHTS.exists():
        raise FileNotFoundError(f"Cannot resume because {LAST_WEIGHTS} does not exist.")
    START_WEIGHTS = LAST_WEIGHTS
    START_LABEL = "resumed 01E"
else:
    available = [path for path in checkpoint_candidates if path.exists()]
    if available:
        START_WEIGHTS = available[0]
        START_LABEL = START_WEIGHTS.parent.parent.name
    else:
        START_WEIGHTS = PROJECT_ROOT / "yolo11n.pt"
        if not START_WEIGHTS.exists():
            START_WEIGHTS = Path("yolo11n.pt")
        START_LABEL = "COCO pretrained YOLO11n"

print(f"Starting model: {START_LABEL}")
print(f"Weights: {START_WEIGHTS}")


## Train

This stage keeps the first ten modules frozen, continues AdamW at a low learning rate,
uses AMP, and applies moderate road-scene augmentation. `cls_pw` weights the
classification loss against BDD100K's severe bus/truck imbalance. The sampler also
rotates scenes without replacement, so short epochs do not mean a tiny static dataset.


In [ ]:
BudgetedTrainer = make_budgeted_detection_trainer(
    samples_per_epoch=SAMPLES_PER_EPOCH,
    balance_power=0.35,
    sampler_seed=SEED,
)

if RUN_TRAINING:
    training_model = YOLO(str(START_WEIGHTS))
    common_train_args = dict(
        trainer=BudgetedTrainer,
        data=str(data_files.main_yaml),
        epochs=EPOCHS,
        patience=40,
        batch=BATCH,
        imgsz=TRAIN_IMGSZ,
        device=DEVICE,
        workers=WORKERS,
        project=str(RUN_ROOT),
        name=TRAIN_RUN_NAME,
        exist_ok=True,
        optimizer="AdamW",
        lr0=0.00025,
        lrf=0.05,
        momentum=0.937,
        weight_decay=0.0006,
        warmup_epochs=1.0,
        freeze=FREEZE,
        amp=True,
        cos_lr=True,
        close_mosaic=15,
        deterministic=False,
        channels_last=True,
        cache=False,
        plots=True,
        save=True,
        save_period=10,
        val=True,
        box=8.0,
        cls=0.65,
        cls_pw=0.35,
        dfl=1.7,
        hsv_h=0.015,
        hsv_s=0.40,
        hsv_v=0.35,
        degrees=0.0,
        translate=0.08,
        scale=0.45,
        shear=0.0,
        perspective=0.0,
        flipud=0.0,
        fliplr=0.5,
        mosaic=0.60,
        mixup=0.02,
        verbose=True,
        seed=SEED,
    )
    if RESUME_TRAINING:
        train_results = training_model.train(resume=True, **common_train_args)
    else:
        train_results = training_model.train(**common_train_args)
    print(f"Training output: {train_results.save_dir}")
else:
    print("RUN_TRAINING=False, skipping training.")


## Inspect training time and learning curves

The median excludes the first epoch when possible because CUDA and data caches warm up
during that epoch. If the median is far from two minutes, scale
`SAMPLES_PER_EPOCH` proportionally and start a new `RUN_TAG`.


In [ ]:
RESULTS_CSV = TRAIN_RUN_DIR / "results.csv"
if RESULTS_CSV.exists():
    history = pd.read_csv(RESULTS_CSV)
    history.columns = [column.strip() for column in history.columns]
    history["epoch_seconds"] = history["time"].diff()
    history.loc[history.index[0], "epoch_seconds"] = history.loc[history.index[0], "time"]
    steady_times = history["epoch_seconds"].iloc[1:] if len(history) > 1 else history["epoch_seconds"]
    median_epoch_seconds = float(steady_times.median())
    recommended_samples = int(
        round(SAMPLES_PER_EPOCH * TARGET_SECONDS_PER_EPOCH / max(median_epoch_seconds, 1.0) / 50) * 50
    )
    recommended_samples = max(100, min(POOL_SIZE, recommended_samples))
    display(history.tail(10))
    print(f"Median steady epoch: {median_epoch_seconds:.1f} seconds")
    print(f"Measured next-run recommendation: {recommended_samples:,} images/epoch")

    figure, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(history["epoch"] + 1, history["metrics/mAP50(B)"], label="mAP50")
    axes[0].plot(history["epoch"] + 1, history["metrics/mAP50-95(B)"], label="mAP50:95")
    axes[0].set_title("Validation AP")
    axes[0].legend()
    axes[1].plot(history["epoch"] + 1, history["metrics/precision(B)"], label="precision")
    axes[1].plot(history["epoch"] + 1, history["metrics/recall(B)"], label="recall")
    axes[1].set_title("Validation precision and recall")
    axes[1].legend()
    axes[2].plot(history["epoch"] + 1, history["epoch_seconds"])
    axes[2].axhline(TARGET_SECONDS_PER_EPOCH, color="black", linestyle="--")
    axes[2].set_title("Seconds per epoch")
    plt.tight_layout()
    plt.show()
else:
    history = pd.DataFrame()
    median_epoch_seconds = float("nan")
    print(f"No training history yet: {RESULTS_CSV}")


## Full 8,000-image no-regression selection

The per-epoch 400-image subset is only a fast progress signal. This cell evaluates the
untouched starting model and the new checkpoint on every validation image at 640
pixels. The model with the highest combined AP score wins, so additional training
cannot silently replace a stronger checkpoint.


In [ ]:
def metric_row(label, weights, metrics):
    precision = float(metrics.box.mp)
    recall = float(metrics.box.mr)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    map50 = float(metrics.box.map50)
    map50_95 = float(metrics.box.map)
    return {
        "candidate": label,
        "weights": str(Path(weights).resolve()),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mAP50": map50,
        "mAP50:95": map50_95,
        "selection_score": map50 + 0.25 * map50_95,
    }

trained_best = TRAIN_RUN_DIR / "weights" / "best.pt"
candidate_paths = {"starting checkpoint": Path(START_WEIGHTS)}
if trained_best.exists() and trained_best.resolve() != Path(START_WEIGHTS).resolve():
    candidate_paths["01E trained"] = trained_best

candidate_rows = []
candidate_metrics = {}
for label, weights in candidate_paths.items():
    print(f"Full validation: {label}")
    model = YOLO(str(weights))
    metrics = model.val(
        data=str(DATA_YAML),
        split="val",
        imgsz=EVAL_IMGSZ,
        batch=EVAL_BATCH,
        device=DEVICE,
        workers=0,
        conf=0.001,
        iou=0.7,
        max_det=300,
        plots=False,
        save_json=False,
        verbose=False,
        project=str(RUN_ROOT / "full_validation"),
        name=f"{RUN_TAG}_{label.replace(' ', '_')}",
        exist_ok=True,
    )
    candidate_metrics[label] = metrics
    candidate_rows.append(metric_row(label, weights, metrics))

candidate_table = pd.DataFrame(candidate_rows).sort_values(
    "selection_score", ascending=False
).reset_index(drop=True)
display(candidate_table.style.format({
    "precision": "{:.3f}",
    "recall": "{:.3f}",
    "f1": "{:.3f}",
    "mAP50": "{:.3f}",
    "mAP50:95": "{:.3f}",
    "selection_score": "{:.3f}",
}))

SELECTED_LABEL = str(candidate_table.iloc[0]["candidate"])
BEST_WEIGHTS = Path(candidate_table.iloc[0]["weights"])
val_metrics = candidate_metrics[SELECTED_LABEL]
best_model = YOLO(str(BEST_WEIGHTS))
print(f"Selected: {SELECTED_LABEL}")
print(f"Selected weights: {BEST_WEIGHTS}")


## Held-out test evaluation

The 2,000-image local test split is used once after model selection. mAP is evaluated
at a low confidence floor so the complete precision-recall curve is measured.


In [ ]:
test_metrics = best_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=EVAL_IMGSZ,
    batch=EVAL_BATCH,
    device=DEVICE,
    workers=0,
    conf=0.001,
    iou=0.7,
    max_det=300,
    plots=True,
    save_json=False,
    verbose=False,
    project=str(RUN_ROOT / "final_test"),
    name=RUN_TAG,
    exist_ok=True,
)

validation_summary = metric_row(SELECTED_LABEL, BEST_WEIGHTS, val_metrics)
validation_summary["split"] = "validation"
test_summary = metric_row(SELECTED_LABEL, BEST_WEIGHTS, test_metrics)
test_summary["split"] = "test"
summary = pd.DataFrame([validation_summary, test_summary])
display(summary[[
    "split", "precision", "recall", "f1", "mAP50", "mAP50:95"
]].style.format({
    "precision": "{:.3f}",
    "recall": "{:.3f}",
    "f1": "{:.3f}",
    "mAP50": "{:.3f}",
    "mAP50:95": "{:.3f}",
}))


## Confidence calibration

`balanced` maximizes per-class F1 for practical detection. `high_confidence_80`
displays only detections with model confidence at least 0.80 and tries to retain a
minimum 0.80 estimated precision per class. Some classes may fail that precision
target or have very low recall; the table reports this explicitly.


In [ ]:
def operating_point(metrics, threshold):
    px = np.asarray(metrics.box.px)
    p_curve = np.asarray(metrics.box.p_curve)
    r_curve = np.asarray(metrics.box.r_curve)
    class_precision = np.array([np.interp(threshold, px, curve) for curve in p_curve])
    class_recall = np.array([np.interp(threshold, px, curve) for curve in r_curve])
    precision = float(np.nanmean(class_precision))
    recall = float(np.nanmean(class_recall))
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    return {
        "confidence": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

def calibrate_profile(metrics, name, score_floor, precision_target=None):
    px = np.asarray(metrics.box.px)
    f1_curve = np.asarray(metrics.box.f1_curve)
    p_curve = np.asarray(metrics.box.p_curve)
    r_curve = np.asarray(metrics.box.r_curve)
    rows = []
    thresholds = {}

    for position, class_id_raw in enumerate(metrics.ap_class_index):
        class_id = int(class_id_raw)
        best_f1_index = int(np.nanargmax(f1_curve[position]))
        selected_index = best_f1_index
        selection = "best F1"
        if precision_target is not None:
            valid = np.flatnonzero(
                (p_curve[position] >= precision_target)
                & (r_curve[position] >= MIN_STRICT_RECALL)
            )
            if len(valid):
                selected_index = int(valid[np.nanargmax(f1_curve[position, valid])])
                selection = "precision target"
            else:
                selection = "target unavailable; confidence-floor fallback"

        threshold = max(score_floor, float(px[selected_index]))
        thresholds[str(class_id)] = threshold
        estimated_precision = float(np.interp(threshold, px, p_curve[position]))
        estimated_recall = float(np.interp(threshold, px, r_curve[position]))
        rows.append({
            "profile": name,
            "class_id": class_id,
            "class": best_model.names[class_id],
            "threshold": threshold,
            "estimated_precision": estimated_precision,
            "estimated_recall": estimated_recall,
            "best_f1": float(f1_curve[position, best_f1_index]),
            "target_met": (
                precision_target is None
                or (
                    estimated_precision >= precision_target
                    and estimated_recall >= MIN_STRICT_RECALL
                )
            ),
            "selection": selection,
        })
    return thresholds, pd.DataFrame(rows)

balanced_thresholds, balanced_calibration = calibrate_profile(
    val_metrics, "balanced", BALANCED_CONFIDENCE_FLOOR
)
strict80_thresholds, strict80_calibration = calibrate_profile(
    val_metrics,
    "high_confidence_80",
    STRICT_CONFIDENCE_FLOOR,
    precision_target=TARGET_STRICT_PRECISION,
)
calibration_table = pd.concat(
    [balanced_calibration, strict80_calibration], ignore_index=True
)
strict_operating_point = operating_point(val_metrics, STRICT_CONFIDENCE_FLOOR)
display(calibration_table.style.format({
    "threshold": "{:.3f}",
    "estimated_precision": "{:.3f}",
    "estimated_recall": "{:.3f}",
    "best_f1": "{:.3f}",
}))
display(pd.DataFrame([strict_operating_point]).style.format("{:.3f}"))


## Save deployment weights and profiles

The real-time command defaults to the 0.80-minimum-confidence profile requested for
the demonstration. Use `--threshold-profile balanced` when recall matters more than
showing only very confident detections.


In [ ]:
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "yolo_two_minute"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DEPLOY_WEIGHTS = OUTPUT_ROOT / "bdd100k_yolo_selected.pt"
DEPLOY_CONFIG = OUTPUT_ROOT / "deployment_config.json"
shutil.copy2(BEST_WEIGHTS, DEPLOY_WEIGHTS)

threshold_profiles = {
    "balanced": {
        "inference_confidence": min(balanced_thresholds.values()),
        "class_thresholds": balanced_thresholds,
    },
    "high_confidence_80": {
        "inference_confidence": min(strict80_thresholds.values()),
        "class_thresholds": strict80_thresholds,
    },
}
deployment = {
    "model_family": "YOLO11",
    "selected_stage": SELECTED_LABEL,
    "weights": str(DEPLOY_WEIGHTS.resolve()),
    "imgsz": EVAL_IMGSZ,
    "max_det": 100,
    "default_threshold_profile": "high_confidence_80",
    "threshold_profiles": threshold_profiles,
    "inference_confidence": threshold_profiles["high_confidence_80"]["inference_confidence"],
    "class_thresholds": strict80_thresholds,
    "class_names": {str(key): value for key, value in best_model.names.items()},
    "minimum_display_confidence": STRICT_CONFIDENCE_FLOOR,
    "target_operating_precision": TARGET_STRICT_PRECISION,
    "strict_operating_point": strict_operating_point,
    "validation_metrics": validation_summary,
    "test_metrics": test_summary,
}
DEPLOY_CONFIG.write_text(json.dumps(deployment, indent=2), encoding="utf-8")
summary.to_csv(OUTPUT_ROOT / "final_metrics.csv", index=False)
calibration_table.to_csv(OUTPUT_ROOT / "confidence_calibration.csv", index=False)

print(f"Weights: {DEPLOY_WEIGHTS}")
print(f"Configuration: {DEPLOY_CONFIG}")
print(
    "Strict live command:\n"
    f'python -m road_detection.realtime_detect --backend yolo '
    f'--weights "{DEPLOY_WEIGHTS}" --config "{DEPLOY_CONFIG}" '
    f'--threshold-profile high_confidence_80 --source 0 --device {DEVICE}'
)


## High-confidence qualitative predictions


In [ ]:
test_images = sorted((PROJECT_ROOT / "data" / "bdd100k_yolo" / "images" / "test").glob("*.jpg"))
sample_images = random.Random(SEED).sample(test_images, min(6, len(test_images)))
inference_confidence = threshold_profiles["high_confidence_80"]["inference_confidence"]
prediction_model = YOLO(str(DEPLOY_WEIGHTS))
predictions = prediction_model.predict(
    sample_images,
    imgsz=EVAL_IMGSZ,
    conf=inference_confidence,
    device=DEVICE,
    quantize=16 if DEVICE != "cpu" else None,
    max_det=100,
    verbose=False,
)
figure, axes = plt.subplots(2, 3, figsize=(16, 9))
for axis, result in zip(axes.flat, predictions):
    axis.imshow(cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB))
    axis.set_title(Path(result.path).name)
    axis.axis("off")
for axis in axes.flat[len(predictions):]:
    axis.axis("off")
plt.tight_layout()
plt.show()


## Real-time speed benchmark


In [ ]:
benchmark_images = test_images[:min(40, len(test_images))]
benchmark_frames = [cv2.imread(str(path)) for path in benchmark_images]
benchmark_frames = [frame for frame in benchmark_frames if frame is not None]
deployment_model = YOLO(str(DEPLOY_WEIGHTS))
deployment_model.fuse()
for frame in benchmark_frames[:5]:
    deployment_model.predict(
        frame,
        imgsz=EVAL_IMGSZ,
        conf=inference_confidence,
        device=DEVICE,
        quantize=16 if DEVICE != "cpu" else None,
        max_det=100,
        verbose=False,
    )
if torch.cuda.is_available():
    torch.cuda.synchronize()

inference_times = []
started = time.perf_counter()
for frame in benchmark_frames:
    result = deployment_model.predict(
        frame,
        imgsz=EVAL_IMGSZ,
        conf=inference_confidence,
        device=DEVICE,
        quantize=16 if DEVICE != "cpu" else None,
        max_det=100,
        verbose=False,
    )[0]
    inference_times.append(float(result.speed["inference"]))
if torch.cuda.is_available():
    torch.cuda.synchronize()
elapsed = time.perf_counter() - started
end_to_end_fps = len(benchmark_frames) / max(elapsed, 1e-9)
model_fps = 1000.0 / max(float(np.mean(inference_times)), 1e-9)
print(f"End-to-end speed: {end_to_end_fps:.2f} FPS")
print(f"Model inference only: {model_fps:.2f} FPS")


## Acceptance decision

These checks keep three different ideas separate:

1. Epoch speed is wall-clock training efficiency.
2. mAP/F1 measure detector quality across the complete validation and test sets.
3. Strict operating precision measures the subset of detections shown at confidence
   0.80 or higher. Raising confidence can increase precision but reduces recall and
   cannot increase mAP.


In [ ]:
validation_row = summary.loc[summary["split"] == "validation"].iloc[0]
test_row = summary.loc[summary["split"] == "test"].iloc[0]
strict_rows = calibration_table[
    calibration_table["profile"] == "high_confidence_80"
]
acceptance = pd.DataFrame([
    {
        "requirement": "median epoch <= 150 seconds",
        "measured": median_epoch_seconds,
        "target": MAX_ACCEPTABLE_SECONDS_PER_EPOCH,
        "passed": (
            np.isfinite(median_epoch_seconds)
            and median_epoch_seconds <= MAX_ACCEPTABLE_SECONDS_PER_EPOCH
        ),
    },
    {
        "requirement": "validation mAP50 >= 0.80",
        "measured": float(validation_row["mAP50"]),
        "target": TARGET_MAP50,
        "passed": float(validation_row["mAP50"]) >= TARGET_MAP50,
    },
    {
        "requirement": "test mAP50 >= 0.80",
        "measured": float(test_row["mAP50"]),
        "target": TARGET_MAP50,
        "passed": float(test_row["mAP50"]) >= TARGET_MAP50,
    },
    {
        "requirement": "validation F1 >= 0.80",
        "measured": float(validation_row["f1"]),
        "target": TARGET_F1,
        "passed": float(validation_row["f1"]) >= TARGET_F1,
    },
    {
        "requirement": "precision at confidence 0.80 >= 0.80",
        "measured": strict_operating_point["precision"],
        "target": TARGET_STRICT_PRECISION,
        "passed": strict_operating_point["precision"] >= TARGET_STRICT_PRECISION,
    },
    {
        "requirement": "all classes meet strict precision and recall",
        "measured": int(strict_rows["target_met"].sum()),
        "target": len(strict_rows),
        "passed": bool(strict_rows["target_met"].all()),
    },
])
display(acceptance.style.format({"measured": "{:.3f}", "target": "{:.3f}"}))

final_report = {
    **deployment,
    "run_mode": RUN_MODE,
    "run_tag": RUN_TAG,
    "samples_per_epoch": SAMPLES_PER_EPOCH,
    "pool_size": len(pool_images),
    "median_epoch_seconds": median_epoch_seconds,
    "end_to_end_fps": end_to_end_fps,
    "model_inference_fps": model_fps,
    "candidate_comparison": candidate_table.to_dict(orient="records"),
    "acceptance": acceptance.to_dict(orient="records"),
}
(OUTPUT_ROOT / "final_evaluation.json").write_text(
    json.dumps(final_report, indent=2), encoding="utf-8"
)

if not acceptance["passed"].all():
    print(
        "One or more measured targets remain unmet. The next defensible step is more "
        "training time or a larger GPU/model, not relabeling confidence as accuracy."
    )
